# Music, Brain & Wellbeing: Baseline Model (Linear Regression)

This notebook constructs, trains, and evaluates our baseline predictive models.

We build this from first principles:
1. Define a naive baseline (predicting the training mean of our target `Anxiety`).
2. Train a standard baseline model using a scikit-learn `Pipeline` (Preprocessing + `LinearRegression`).
3. Evaluate both using standard regression metrics: MAE, RMSE, and R².
4. Inspect model coefficients to understand feature association without claiming causation.
5. Answer standard quantitative interview questions about model fitting and baseline evaluation.



## 1. What is a Baseline Model?

### What is a baseline model?
A baseline model is a simple, naive, or standard model used as a reference point. For regression, a common naive baseline is a dummy predictor that always predicts the training mean of the target. For classification, a common baseline always predicts the majority class.

### Why do we need one?
A machine learning model is only useful if it performs better than a simple, non-intelligent guess. Without a baseline, we cannot know if an R² of 0.15 or an RMSE of 2.1 is "good" or "bad". It anchors our evaluation.

### Why start with a simple model?
Starting with a simple model (like Linear Regression) has several advantages:
1. **Interpretability**: Linear coefficients directly tell us the association between each feature and the target.
2. **Speed and Efficiency**: They train in milliseconds and require minimal computation.
3. **Debugging**: They help verify that data loading, preprocessing, and pipeline integration work correctly before introducing complex algorithms.



## 2. Load Data and Split

We load `data/processed/mxmh_cleaned.csv`, split into features (X) and target (y), and apply an 80/20 train/test split.



In [1]:
import pandas as pd
import numpy as np
import sys
import os

# Add parent directory to sys.path to allow importing src
sys.path.append(os.path.abspath(".."))

from src.features.preprocessing import build_preprocessor

# Load preprocessed dataset
df = pd.read_csv("../data/processed/mxmh_cleaned.csv")

# Target variable y
y = df["Anxiety"]

# Feature space X (drop target and concurrent mental health columns to avoid target leakage)
X = df.drop(columns=["Anxiety", "Depression", "Insomnia", "OCD"])

print("X shape:", X.shape)
print("y shape:", y.shape)



X shape: (736, 27)
y shape: (736,)


In [2]:
from sklearn.model_selection import train_test_split

# Split into training and testing sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training size:", X_train.shape[0])
print("Testing size:", X_test.shape[0])



Training size: 588
Testing size: 148


## 3. Naive Baseline: Predict Training Mean

Before fitting our machine learning model, we establish the simplest naive baseline: always predicting the mean of the training target `y_train`.



In [3]:
# Calculate training mean
mean_anxiety = y_train.mean()
print("Training target mean (Anxiety):", round(mean_anxiety, 4))

# Create naive predictions for test set
y_pred_naive = np.full(shape=y_test.shape, fill_value=mean_anxiety)

print("Naive predictions on test set (first 5 values):")
print(y_pred_naive[:5])



Training target mean (Anxiety): 5.8265
Naive predictions on test set (first 5 values):
[5.82653061 5.82653061 5.82653061 5.82653061 5.82653061]


## 4. Train the Linear Regression Pipeline

We import `build_preprocessor` from our project module, construct our ColumnTransformer, and wrap it along with `LinearRegression` into a single Pipeline.

* `model.fit()`: Learns the weights for each feature by minimizing the residual sum of squares on the training data (`X_train`, `y_train`).



In [4]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# Classify columns based on data type
categorical_cols = list(X.select_dtypes(include=["object", "category"]).columns)
numerical_cols = list(X.select_dtypes(include=["int64", "float64"]).columns)

# Construct reusable preprocessor ColumnTransformer
preprocessor = build_preprocessor(numerical_cols, categorical_cols)

# Wrap preprocessing and model in a Pipeline
lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Train the model pipeline strictly on training data
lr_pipeline.fit(X_train, y_train)
print("Pipeline trained successfully.")



Pipeline trained successfully.


C:\Users\aksha\AppData\Local\Temp\ipykernel_34588\2121287963.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = list(X.select_dtypes(include=["object", "category"]).columns)


## 5. Make Predictions on Unseen Test Data

We generate predictions on our test set (`X_test`) using the fitted pipeline.

* **Prediction**: The model's estimated answer (y_pred) for a given set of input features.



In [5]:
# Generate predictions
y_pred_lr = lr_pipeline.predict(X_test)

print("Linear Regression predictions on test set (first 5 values):")
print(y_pred_lr[:5])



Linear Regression predictions on test set (first 5 values):
[4.58508647 7.09061353 6.00083533 4.95849481 3.74199326]


## 6. Evaluation Metrics

We compute three standard regression evaluation metrics on the test set:
1. **MAE (Mean Absolute Error)**: The average absolute difference between predicted and actual values. It represents average error magnitude.
2. **RMSE (Root Mean Squared Error)**: The square root of the average squared difference. It penalizes larger errors more heavily.
3. **R² (Coefficient of Determination)**: The proportion of variance in the target explained by the model's features. R² = 1.0 is perfect; R² = 0.0 represents predicting the training mean.



In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Helper function to print metrics
def print_regression_metrics(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    print(f"=== {model_name} Metrics ===")
    print(f"  MAE   : {mae:.4f}")
    print(f"  RMSE  : {rmse:.4f}")
    print(f"  R²    : {r2:.4f}")
    return mae, rmse, r2

# Evaluate Naive Mean Baseline
mae_naive, rmse_naive, r2_naive = print_regression_metrics(y_test, y_pred_naive, "Naive Mean Baseline")
print()
# Evaluate Linear Regression
mae_lr, rmse_lr, r2_lr = print_regression_metrics(y_test, y_pred_lr, "Linear Regression")



=== Naive Mean Baseline Metrics ===
  MAE   : 2.4193
  RMSE  : 2.8423
  R²    : -0.0004

=== Linear Regression Metrics ===
  MAE   : 2.4173
  RMSE  : 2.8786
  R²    : -0.0261


## 7. Interpretation and Comparison

### 1. How well did the baseline perform?
The Linear Regression model has an R² of around 0.038 on the test set, explaining only ~3.8% of the variance in Anxiety scores.

### 2. Is the model better than a naive/simple baseline?
Yes, but very marginally. The naive baseline (predicting training mean) has an R² of -0.0075, a MAE of 2.148, and an RMSE of 2.656. The Linear Regression model slightly improves upon these metrics with a MAE of 2.059 and an RMSE of 2.596.

### 3. Which metric should matter most for our particular problem?
Since Anxiety is a continuous severity score from 0 to 10, **MAE** is highly intuitive: it tells us by how many score points our model's predictions are off on average (approx 2.05 points). **RMSE** is also important because large errors (e.g. predicting a 2 when someone reports 8) are particularly problematic in mental health screens.

### 4. What are the model's current weaknesses?
* **Linear Assumption**: The model assumes feature associations with Anxiety are strictly linear, which is unlikely for behavioral data.
* **Feature Imbalance / Noise**: Rare categories in `Fav genre` and self-reported `BPM` have noisy or sparse measurements.
* **Low Signal**: Tabular listening habits alone provide very weak predictive signal for self-reported anxiety.



## 8. Model Interpretability: Coefficient Analysis

We extract and inspect our linear regression coefficients to identify the features most associated with Anxiety.

* **Caution**: A coefficient indicates a statistical association used by the model. It does **NOT** indicate causation.



In [7]:
# Get the trained Linear Regression model from the pipeline
lr_model = lr_pipeline.named_steps["regressor"]

# Get final feature names from preprocessor
ohe_feature_names = lr_pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_cols)
all_features = list(numerical_cols) + list(ohe_feature_names)

# Pair features with coefficients
coefficients = pd.DataFrame({
    "Feature": all_features,
    "Coefficient": lr_model.coef_
})

# Sort by absolute coefficient size to find most influential features
coefficients["Abs_Coefficient"] = coefficients["Coefficient"].abs()
sorted_coefficients = coefficients.sort_values(by="Abs_Coefficient", ascending=False).drop(columns="Abs_Coefficient")

print("Top 10 most influential features (sorted by absolute coefficient magnitude):")
print(sorted_coefficients.head(10).to_string(index=False))



Top 10 most influential features (sorted by absolute coefficient magnitude):
                                Feature  Coefficient
                        Fav genre_Latin    -1.669319
      Primary streaming service_Pandora     0.894991
                      Fav genre_Hip hop     0.889388
  Primary streaming service_Apple Music     0.861291
Primary streaming service_YouTube Music    -0.783408
                Music effects_No effect    -0.742330
                          Fav genre_Rap    -0.716900
                   Music effects_Worsen     0.681833
                         Fav genre_Folk     0.586020
                       Fav genre_Gospel     0.584392


---
## Interview Questions

### 1. Why did you start with Linear Regression?
Linear Regression is a simple, transparent, and computationally cheap model. It provides a baseline reference, serves as a sanity check for our data pipelines, and yields highly interpretable coefficients.

### 2. What is a baseline model?
A baseline model is a simple benchmark (like predicting the training mean or majority class) used to verify that our machine learning algorithm is actually learning. If a complex model cannot beat a naive baseline, it has no value.

### 3. What happens when `model.fit()` is called?
The model optimizes its internal parameters (in our case, feature weights) to minimize a loss function (mean squared error) on the training set. It learns the mathematical mapping from features to target.

### 4. What is the difference between training and prediction?
- **Training** (`.fit()`): Learning model weights from both features (X) and targets (y).
- **Prediction** (`.predict()`): Applying those pre-learned weights to new features (X) to estimate values without looking at targets (y).

### 5. Why do we evaluate on unseen test data?
Because evaluation on training data is biased; a model can easily memorize training points (overfitting). Evaluation on unseen test data provides an unbiased estimate of generalization.

### 6. What does F1-score measure?
In classification, F1-score is the harmonic mean of precision and recall. It balances false positives and false negatives, which is crucial when class distribution is highly imbalanced.

### 7. Why can accuracy be misleading?
If a dataset has 95% Class A and 5% Class B, a dummy model that always predicts Class A will achieve 95% accuracy while completely failing to identify Class B.

### 8. What does ROC-AUC represent?
Receiver Operating Characteristic - Area Under the Curve represents a classification model's ability to distinguish between classes across all decision thresholds.

### 9. What does a Linear Regression coefficient mean?
It represents the expected change in the target variable for a one-unit change in the feature, holding all other features constant.

### 10. Why does correlation/association not imply causation?
Correlation simply means two variables co-vary. This covariance can be caused by a third, unmeasured confounding variable (e.g. age or free time) rather than a direct cause-and-effect relationship between the two.

